# Returning allergen info

Here we want to try to look at inputs being a select list of allergens from a list. So lets say we have bronopol - https://www.contactdermatitisinstitute.com/2-bromo-2nitropropane-1-3-diol-bronopol.php

We select 'Bronopol' from our list but the ingredients taken from OCR are instead: '2-Bromo-2-nitropropane-1,3-diol' (i.e. an alternative name)

How do we now make sure that when we select 'Bronopol' that we then match to all possible versions of it's name?

1. Select 'Bronopol' on multi-select
2. Return the entire list it belongs to (because these have all the alternative names)
3. Display 'Allergens detected:' then top level allergen (Bronopol) and subset under that as bullet point we'd have '2-Bromo-2-nitropropane-1,3-diol'

## Load packages and check working directory

In [102]:
import easyocr
import cv2
from matplotlib import pyplot as plt
import re
import os
import json

current_dir = os.path.abspath('.')
parent_dir = os.path.dirname(current_dir)
print(f"Current directory: {current_dir}")
print(f"Parent directory: {parent_dir}")

Current directory: c:\Users\SamHenderson-Palmer\Documents\Github\00.Personal\ocr-ingredients-checker\notebooks
Parent directory: c:\Users\SamHenderson-Palmer\Documents\Github\00.Personal\ocr-ingredients-checker


## Set fixed paths and variables

In [103]:
MODEL_PATH = f'{parent_dir}/models/easyocr'
IMAGE_PATH = f'{parent_dir}/assets/sample_images/qv_gentlewash.jpg'
ALLERGEN_LIST_PATH = f'{parent_dir}/assets/data/allergens_cleaned.json'
DELIMITERS = {",", ";", ":"}

with open(ALLERGEN_LIST_PATH, "r", encoding="utf-8") as file:
    ALLERGEN_LIST = json.load(file)

In [104]:
allergen_names = [item["item_name"] for item in ALLERGEN_LIST]
allergen_names

['(nitrobutyl) morpholine / (ethylnitro-trimethylene) dimorpholi',
 '1,2-benzisothiazoline-3-one, sodium salt',
 '1,3-butandiol-dimethacrylate',
 '1,3-diphenylguanidine',
 '1,3,5-tris-(2-hydroxyethyl)-hexahydrotriazine (grotan bk)',
 '1,4-butandioldimethacrylat (budma)',
 '1,4-butanedioldiglycidyl ether',
 '1,6-hexanediolediglycidyl ether',
 '2-(2-aminoethoxy)-eth',
 '2-bromo-2nitropropane-1, 3-diol (bronopol)',
 '2-ethylhexyl acrylate',
 '2-ethylhexyl-4-dimethylaminobenzoate',
 '2-ethylhexyl-p-methoxycinnamate',
 '2-ethylhexyl-p-methoxycinnamate (octinoxate)',
 '2-hydroxy-4-methoxy-benzophenone',
 '2-hydroxy-ethylacrylate',
 '2-hydroxymethyl-2-nitro-1,3-propanediol',
 '2-hydroxypropyl-methacrylate',
 '2-mercaptobenzimidazole',
 '2-mercaptobenzothiazole',
 '3-(4-methylbenzylidene) camphor',
 '3-aminophenol',
 '3,4,4-triclocarban',
 '4-aminoazobenzene',
 '4-chloro-3-cresol (pcmc)',
 '4-chloro-3,5-xylenol (pcmx)',
 '4-hexyl-resorcinol',
 '4-phenylenediamine base',
 '4-tert-butyl-4-methox

In [105]:
selected_allergens = ['2-bromo-2nitropropane-1, 3-diol (bronopol)', 'butylacrylate']

We can return the list of alternative names that corresponds to our selected allergen

In [106]:
data = [item for item in ALLERGEN_LIST if item["item_name"] in selected_allergens]
data

[{'item_name': '2-bromo-2nitropropane-1, 3-diol (bronopol)',
  'url': 'https://www.contactdermatitisinstitute.com/2-bromo-2nitropropane-1-3-diol-bronopol.php',
  'allergens': ['2-Bromo-2-nitropropane-1,3-diol',
   '2-Nitro-2-bromo-1,3-propanediol',
   'Bronidiol',
   'Bronocot',
   'Bronopol',
   'Bronosol',
   'Bronotak',
   'HSDB 7195',
   'Lexgard bronopol',
   'NSC 141021',
   'Onyxide 500',
   'beta-Bromo-beta-nitrotrimethyleneglycol Germall® 11']},
 {'item_name': 'butylacrylate',
  'url': 'https://www.contactdermatitisinstitute.com/butylacrylate.php',
  'allergens': ['2-Propenoic acid, butyl ester',
   '4-02-00-01463 (Beilstein HandbookReference)',
   'AI3-15739',
   'Acrylic acid n-butyl ester',
   'Acrylic acid, butyl ester',
   'BRN 1749970',
   'Butyl 2-propenoate',
   'Butylester kyseliny akrylove',
   'Butylester kyseliny akrylove [Czech]',
   'CCRIS 3401',
   'EINECS 205-480-7',
   'HSDB 305',
   'NSC 5163',
   'n-Butyl acrylate',
   'n-Butyl propenoate']}]

We now need to normalise the ingredients list of each so it can be checked against normalised ingredients list from OCR

In [107]:
for item in data:
    item["allergens"] = [a.lower().strip() for a in item["allergens"]]

In [108]:
data

[{'item_name': '2-bromo-2nitropropane-1, 3-diol (bronopol)',
  'url': 'https://www.contactdermatitisinstitute.com/2-bromo-2nitropropane-1-3-diol-bronopol.php',
  'allergens': ['2-bromo-2-nitropropane-1,3-diol',
   '2-nitro-2-bromo-1,3-propanediol',
   'bronidiol',
   'bronocot',
   'bronopol',
   'bronosol',
   'bronotak',
   'hsdb 7195',
   'lexgard bronopol',
   'nsc 141021',
   'onyxide 500',
   'beta-bromo-beta-nitrotrimethyleneglycol germall® 11']},
 {'item_name': 'butylacrylate',
  'url': 'https://www.contactdermatitisinstitute.com/butylacrylate.php',
  'allergens': ['2-propenoic acid, butyl ester',
   '4-02-00-01463 (beilstein handbookreference)',
   'ai3-15739',
   'acrylic acid n-butyl ester',
   'acrylic acid, butyl ester',
   'brn 1749970',
   'butyl 2-propenoate',
   'butylester kyseliny akrylove',
   'butylester kyseliny akrylove [czech]',
   'ccris 3401',
   'einecs 205-480-7',
   'hsdb 305',
   'nsc 5163',
   'n-butyl acrylate',
   'n-butyl propenoate']}]

Now we need to work out a way of checking for matches. We could experiment by looking at what if 'bronidiol' is returned in OCR ingredients and we selected '2-bromo-2nitropropane-1, 3-diol (bronopol)' as our allergen. What we want is our same list above but only returning the matches.

In [109]:
ocr_ingredients = [{'ingredient_text': 'lngredlenis', 'words': [{'raw_text': 'LNGREDLENIS:', 'top_left': (159, 244), 'bottom_right': (334, 281)}]}, {'ingredient_text': 'iie gaycolvozok copolymer', 'words': [{'raw_text': 'Iie', 'top_left': (0, 224), 'bottom_right': (130, 344)}, {'raw_text': 'GaycolVOzok', 'top_left': (158, 269), 'bottom_right': (283, 324)}, {'raw_text': 'Copolymer,', 'top_left': (282, 277), 'bottom_right': (396, 309)}]}, {'ingredient_text': 'petrolatum', 'words': [{'raw_text': 'Petrolatum,', 'top_left': (400, 280), 'bottom_right': (520, 306)}]}, {'ingredient_text': 'alcohol denat', 'words': [{'raw_text': 'Alcohol', 'top_left': (522, 280), 'bottom_right': (600, 306)}, {'raw_text': 'Denat;', 'top_left': (600, 278), 'bottom_right': (664, 304)}]}, {'ingredient_text': 'pvp_ copezda ozokerite', 'words': [{'raw_text': 'PVP_', 'top_left': (666, 274), 'bottom_right': (708, 300)}, {'raw_text': 'Copezda', 'top_left': (705, 262), 'bottom_right': (796, 325)}, {'raw_text': 'Ozokerite,', 'top_left': (224, 296), 'bottom_right': (325, 329)}]}, {'ingredient_text': 'anhydrous lanolin', 'words': [{'raw_text': 'Anhydrous', 'top_left': (326, 304), 'bottom_right': (436, 332)}, {'raw_text': 'Lanolin,', 'top_left': (436, 306), 'bottom_right': (522, 332)}]}, {'ingredient_text': 'cera alba beeswax', 'words': [{'raw_text': 'Cera', 'top_left': (524, 306), 'bottom_right': (574, 330)}, {'raw_text': 'Alba', 'top_left': (576, 304), 'bottom_right': (622, 328)}, {'raw_text': '(Beeswax) ,', 'top_left': (620, 297), 'bottom_right': (711, 327)}]}, {'ingredient_text': 'cera carnauba', 'words': [{'raw_text': 'Cera', 'top_left': (232, 322), 'bottom_right': (278, 348)}, {'raw_text': '(Carnauba) ,', 'top_left': (277, 323), 'bottom_right': (397, 356)}]}, {'ingredient_text': 'peg-30 glyceryl stearate', 'words': [{'raw_text': 'PEG-30', 'top_left': (398, 330), 'bottom_right': (482, 356)}, {'raw_text': 'Glyceryl', 'top_left': (483, 329), 'bottom_right': (570, 358)}, {'raw_text': 'Stearate,', 'top_left': (568, 328), 'bottom_right': (650, 352)}]}, {'ingredient_text': 'polyacrylamide', 'words': [{'raw_text': 'Polyacrylamide,', 'top_left': (161, 338), 'bottom_right': (298, 378)}]}, {'ingredient_text': 'peg-8 beeswax', 'words': [{'raw_text': 'PEG-8', 'top_left': (299, 349), 'bottom_right': (372, 378)}, {'raw_text': 'Beeswax,', 'top_left': (370, 353), 'bottom_right': (465, 383)}]}, {'ingredient_text': 'c13-14 isoparaffin', 'words': [{'raw_text': 'C13-14', 'top_left': (464, 354), 'bottom_right': (544, 380)}, {'raw_text': 'isoparaffin;', 'top_left': (541, 346), 'bottom_right': (653, 385)}]}, {'ingredient_text': 'oil', 'words': [{'raw_text': 'Oil;', 'top_left': (714, 364), 'bottom_right': (748, 388)}]}, {'ingredient_text': 'disodium edta argania spinosa kernal argan dv ethylhexylglycerin', 'words': [{'raw_text': 'Disodium', 'top_left': (253, 371), 'bottom_right': (351, 401)}, {'raw_text': 'EDTA', 'top_left': (351, 375), 'bottom_right': (422, 404)}, {'raw_text': 'Argania', 'top_left': (430, 380), 'bottom_right': (508, 408)}, {'raw_text': 'Spinosa', 'top_left': (508, 378), 'bottom_right': (586, 408)}, {'raw_text': 'Kernal', 'top_left': (586, 371), 'bottom_right': (651, 401)}, {'raw_text': '(Argan)', 'top_left': (650, 370), 'bottom_right': (716, 398)}, {'raw_text': 'DV', 'top_left': (846, 358), 'bottom_right': (872, 402)}, {'raw_text': 'Ethylhexylglycerin,', 'top_left': (247, 393), 'bottom_right': (434, 437)}]}, {'ingredient_text': 'acrylamide', 'words': [{'raw_text': 'Acrylamide,', 'top_left': (433, 401), 'bottom_right': (552, 432)}]}, {'ingredient_text': 'citral', 'words': [{'raw_text': 'Citral,', 'top_left': (554, 400), 'bottom_right': (616, 426)}]}, {'ingredient_text': 'extract', 'words': [{'raw_text': 'Extract;', 'top_left': (314, 422), 'bottom_right': (395, 453)}]}, {'ingredient_text': 'geraniol', 'words': [{'raw_text': 'Geraniol,', 'top_left': (396, 428), 'bottom_right': (490, 452)}]}, {'ingredient_text': 'limonene', 'words': [{'raw_text': 'Limonene,', 'top_left': (491, 429), 'bottom_right': (595, 449)}]}, {'ingredient_text': 'linalool aqua', 'words': [{'raw_text': 'Linalool.', 'top_left': (594, 417), 'bottom_right': (675, 447)}, {'raw_text': 'Aqua;', 'top_left': (163, 268), 'bottom_right': (211, 298)}]}, {'ingredient_text': 'cerifera parfum', 'words': [{'raw_text': 'Cerifera', 'top_left': (165, 312), 'bottom_right': (229, 346)}, {'raw_text': 'Parfum,', 'top_left': (648, 322), 'bottom_right': (718, 343)}]}, {'ingredient_text': 'lhsh phenoxylethanol', 'words': [{'raw_text': 'LHSH', 'top_left': (0, 319), 'bottom_right': (76, 392)}, {'raw_text': 'Phenoxylethanol;', 'top_left': (648, 347), 'bottom_right': (782, 361)}]}, {'ingredient_text': 'laureth-7 coumarin', 'words': [{'raw_text': 'Laureth-7', 'top_left': (165, 359), 'bottom_right': (246, 397)}, {'raw_text': 'Coumarin;', 'top_left': (712, 387), 'bottom_right': (790, 404)}]}, {'ingredient_text': 'bronopol', 'words': [{'raw_text': 'Bronopol,', 'top_left': (165, 383), 'bottom_right': (247, 423)}]}, {'ingredient_text': 'citronellol', 'words': [{'raw_text': 'Citronellol;', 'top_left': (615, 396), 'bottom_right': (714, 414)}]}, {'ingredient_text': 'evernia prunastri', 'words': [{'raw_text': 'Evernia', 'top_left': (167, 405), 'bottom_right': (227, 441)}, {'raw_text': 'Prunastri', 'top_left': (231, 413), 'bottom_right': (313, 451)}]}]

In [110]:
ocr_ingredients

[{'ingredient_text': 'lngredlenis',
  'words': [{'raw_text': 'LNGREDLENIS:',
    'top_left': (159, 244),
    'bottom_right': (334, 281)}]},
 {'ingredient_text': 'iie gaycolvozok copolymer',
  'words': [{'raw_text': 'Iie',
    'top_left': (0, 224),
    'bottom_right': (130, 344)},
   {'raw_text': 'GaycolVOzok',
    'top_left': (158, 269),
    'bottom_right': (283, 324)},
   {'raw_text': 'Copolymer,',
    'top_left': (282, 277),
    'bottom_right': (396, 309)}]},
 {'ingredient_text': 'petrolatum',
  'words': [{'raw_text': 'Petrolatum,',
    'top_left': (400, 280),
    'bottom_right': (520, 306)}]},
 {'ingredient_text': 'alcohol denat',
  'words': [{'raw_text': 'Alcohol',
    'top_left': (522, 280),
    'bottom_right': (600, 306)},
   {'raw_text': 'Denat;',
    'top_left': (600, 278),
    'bottom_right': (664, 304)}]},
 {'ingredient_text': 'pvp_ copezda ozokerite',
  'words': [{'raw_text': 'PVP_',
    'top_left': (666, 274),
    'bottom_right': (708, 300)},
   {'raw_text': 'Copezda', 'top

In [111]:
for ingredient in ocr_ingredients:
    print(ingredient["ingredient_text"])

lngredlenis
iie gaycolvozok copolymer
petrolatum
alcohol denat
pvp_ copezda ozokerite
anhydrous lanolin
cera alba beeswax
cera carnauba
peg-30 glyceryl stearate
polyacrylamide
peg-8 beeswax
c13-14 isoparaffin
oil
disodium edta argania spinosa kernal argan dv ethylhexylglycerin
acrylamide
citral
extract
geraniol
limonene
linalool aqua
cerifera parfum
lhsh phenoxylethanol
laureth-7 coumarin
bronopol
citronellol
evernia prunastri


So we basically want to see for each ocr ingredient; is there a match for any of our selected allergens with their alternative names? If there is then make a note that it's a match and also a note of the name of which selected allergen it's under (plus URL).

In [113]:
result = []

for ingredient in ocr_ingredients:
    # Does any user defined allergen match this ingredient
    matched_allergens = []
    parent_allergens = []

    for allergen_overview in data:
        parent_name = allergen_overview["item_name"]
        related_names = allergen_overview["allergens"]
        url = allergen_overview["url"]

        for allergen in related_names:
            if allergen in ingredient["ingredient_text"]:
                matched_allergens.append(allergen.title())
                parent_allergens.append(parent_name)

    new_ingredient = ingredient.copy()
    new_ingredient["matched_allergens"] = matched_allergens
    new_ingredient["parent_allergen"] = parent_allergens
    new_ingredient["is_match"] = len(matched_allergens) > 0

    result.append(new_ingredient)

In [114]:
result

[{'ingredient_text': 'lngredlenis',
  'words': [{'raw_text': 'LNGREDLENIS:',
    'top_left': (159, 244),
    'bottom_right': (334, 281)}],
  'matched_allergens': [],
  'parent_allergen': [],
  'is_match': False},
 {'ingredient_text': 'iie gaycolvozok copolymer',
  'words': [{'raw_text': 'Iie',
    'top_left': (0, 224),
    'bottom_right': (130, 344)},
   {'raw_text': 'GaycolVOzok',
    'top_left': (158, 269),
    'bottom_right': (283, 324)},
   {'raw_text': 'Copolymer,',
    'top_left': (282, 277),
    'bottom_right': (396, 309)}],
  'matched_allergens': [],
  'parent_allergen': [],
  'is_match': False},
 {'ingredient_text': 'petrolatum',
  'words': [{'raw_text': 'Petrolatum,',
    'top_left': (400, 280),
    'bottom_right': (520, 306)}],
  'matched_allergens': [],
  'parent_allergen': [],
  'is_match': False},
 {'ingredient_text': 'alcohol denat',
  'words': [{'raw_text': 'Alcohol',
    'top_left': (522, 280),
    'bottom_right': (600, 306)},
   {'raw_text': 'Denat;', 'top_left': (600